# La Liga Final Positions Prediction

This notebook predicts final league position probabilities using:
- Dixon-Coles match model
- Elo ratings
- Optional bookmaker odds blending
- Monte Carlo season simulation

Requirements: `FOOTBALL_DATA_API_KEY` in `.env` (optional: `ODDS_API_KEY`).


In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv

from standing_prediction.data import FootballDataClient, fetch_matches, fetch_standings
from standing_prediction.dixon_coles import fit_dixon_coles
from standing_prediction.elo import fit_elo
from standing_prediction.odds import (
    COMPETITION_TO_FOOTBALL_DATA_UK_LEAGUE,
    fetch_the_odds_api_h2h_probs,
    fetch_football_data_uk_probs,
)
from standing_prediction.predict_positions import build_fixture_specs
from standing_prediction.simulate import simulate_season_scores

load_dotenv('.env')


In [ ]:
competition = "PD"  # La Liga

now = pd.Timestamp.utcnow()
season_start = int(now.year if now.month >= 7 else now.year - 1)

N_SIM = 3000
SEED = 7
MAX_GOALS = 10
XI = 0.003
LAMBDA_REG = 0.1
WEIGHTS = {"main": 0.35, "odds": 0.55, "elo": 0.10}


In [ ]:
client = FootballDataClient()

standings = fetch_standings(client, competition=competition, season=season_start)
finished = fetch_matches(client, competition=competition, season=season_start, status="FINISHED")
remaining = fetch_matches(client, competition=competition, season=season_start, status="SCHEDULED,TIMED")


In [ ]:
dc_model = fit_dixon_coles(finished, xi=XI, lambda_reg=LAMBDA_REG)
elo_model = fit_elo(finished)


In [ ]:
odds_df = None
api_key = os.getenv("ODDS_API_KEY") or os.getenv("ODDS_DATA_API_KEY")

if api_key:
    try:
        odds_df = fetch_the_odds_api_h2h_probs(
            sport_key="soccer_spain_la_liga",
            api_key=api_key,
            days=180,
        )
    except Exception:
        odds_df = None

if odds_df is None:
    try:
        league_code = COMPETITION_TO_FOOTBALL_DATA_UK_LEAGUE[competition]
        odds_df = fetch_football_data_uk_probs(
            league_code=league_code,
            season_start_year=season_start,
        )
    except Exception:
        odds_df = None


In [ ]:
fixture_specs = build_fixture_specs(
    remaining,
    dc=dc_model,
    elo=elo_model,
    odds=odds_df,
    max_goals=MAX_GOALS,
    weights=WEIGHTS,
    calibrator=None,
)


In [ ]:
counts = simulate_season_scores(
    fixture_specs,
    standings,
    n_sim=N_SIM,
    seed=SEED,
)

position_pct = counts.div(counts.sum(axis=1), axis=0) * 100
teams_ordered = standings.sort_values("position")["team"]
position_pct = position_pct.loc[teams_ordered]


In [8]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

greens = plt.cm.Greens
green_cmap = LinearSegmentedColormap.from_list(
    "Greens_soft",
    greens(np.linspace(0.05, 0.72, 256)),
)

vmax = 20

def zero_style(val):
    if val < 0.005:
        return "background-color: white !important;"
    return ""

position_pct.style.background_gradient(cmap=green_cmap, vmin=0, vmax=vmax)\
    .map(zero_style)\
    .format("{:.2f}%")\
    .set_table_styles([
        {"selector": "th", "props": [("text-align", "center"), ("font-size", "13px")]},
        {"selector": "th.row_heading", "props": [("text-align", "left"), ("font-size", "13px")]},
        {"selector": "td", "props": [("text-align", "center"), ("font-size", "12px")]}
    ])


,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20
FC Barcelona,59.97%,39.07%,0.90%,0.07%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Real Madrid CF,39.90%,58.70%,1.37%,0.03%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Villarreal CF,0.13%,2.07%,75.37%,20.53%,1.80%,0.10%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Club Atlético de Madrid,0.00%,0.17%,20.73%,63.20%,14.93%,0.87%,0.10%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
Real Betis Balompié,0.00%,0.00%,1.63%,14.93%,65.63%,13.27%,3.30%,0.87%,0.23%,0.13%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
RC Celta de Vigo,0.00%,0.00%,0.00%,0.97%,10.77%,41.57%,24.17%,11.83%,5.43%,2.97%,1.43%,0.53%,0.20%,0.07%,0.03%,0.00%,0.03%,0.00%,0.00%,0.00%
RCD Espanyol de Barcelona,0.00%,0.00%,0.00%,0.00%,1.07%,9.37%,15.57%,19.93%,17.07%,12.43%,10.27%,5.80%,3.73%,2.00%,1.53%,0.83%,0.23%,0.17%,0.00%,0.00%
Athletic Club,0.00%,0.00%,0.00%,0.03%,0.30%,3.43%,6.77%,10.23%,13.87%,15.70%,13.37%,11.30%,9.13%,7.33%,4.13%,2.67%,1.20%,0.43%,0.10%,0.00%
CA Osasuna,0.00%,0.00%,0.00%,0.23%,3.40%,17.53%,24.80%,19.77%,14.70%,8.50%,5.20%,3.00%,1.40%,0.73%,0.43%,0.17%,0.10%,0.03%,0.00%,0.00%
Real Sociedad de Fútbol,0.00%,0.00%,0.00%,0.00%,1.90%,11.43%,17.27%,19.70%,17.67%,12.50%,7.00%,5.17%,2.90%,2.47%,0.87%,0.57%,0.43%,0.10%,0.03%,0.00%
